In [1]:
"""
DQN from scratch (PyTorch) on a custom GridWorld (no Gym).

pip install torch numpy
Run: python dqn_gridworld.py
"""

import math
import random
from dataclasses import dataclass
from typing import Deque, Tuple, List

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim


# -----------------------------
# 1) Tiny GridWorld (from scratch)
# -----------------------------
@dataclass
class GridWorld:
    rows: int = 5
    cols: int = 5
    start: Tuple[int, int] = (0, 0)
    goal: Tuple[int, int] = (4, 4)
    pit: Tuple[int, int] = (3, 3)
    step_cost: float = -0.01
    max_steps: int = 200

    # Actions: 0=UP, 1=RIGHT, 2=DOWN, 3=LEFT
    n_actions: int = 4

    def __post_init__(self):
        self.reset()

    def reset(self) -> np.ndarray:
        self.pos = self.start
        self.t = 0
        return self._obs()

    def _obs(self) -> np.ndarray:
        """Continuous state features for NN: normalized (row, col)."""
        r, c = self.pos
        rr = 0.0 if self.rows == 1 else r / (self.rows - 1)
        cc = 0.0 if self.cols == 1 else c / (self.cols - 1)
        return np.array([rr, cc], dtype=np.float32)

    def step(self, action: int) -> Tuple[np.ndarray, float, bool, dict]:
        r, c = self.pos
        self.t += 1

        if action == 0:   # UP
            r = max(0, r - 1)
        elif action == 1: # RIGHT
            c = min(self.cols - 1, c + 1)
        elif action == 2: # DOWN
            r = min(self.rows - 1, r + 1)
        elif action == 3: # LEFT
            c = max(0, c - 1)
        else:
            raise ValueError("Invalid action")

        self.pos = (r, c)

        done = False
        reward = float(self.step_cost)

        if self.pos == self.goal:
            reward = 1.0
            done = True
        elif self.pos == self.pit:
            reward = -1.0
            done = True
        elif self.t >= self.max_steps:
            done = True

        return self._obs(), reward, done, {}


# -----------------------------
# 2) Replay Buffer (from scratch)
# -----------------------------
class ReplayBuffer:
    def __init__(self, capacity: int, state_dim: int):
        self.capacity = int(capacity)
        self.state_dim = int(state_dim)
        self.ptr = 0
        self.size = 0

        self.states = np.zeros((capacity, state_dim), dtype=np.float32)
        self.actions = np.zeros((capacity,), dtype=np.int64)
        self.rewards = np.zeros((capacity,), dtype=np.float32)
        self.next_states = np.zeros((capacity, state_dim), dtype=np.float32)
        self.dones = np.zeros((capacity,), dtype=np.float32)  # 1.0 if done else 0.0

    def push(self, s, a, r, s2, done):
        i = self.ptr
        self.states[i] = s
        self.actions[i] = a
        self.rewards[i] = r
        self.next_states[i] = s2
        self.dones[i] = 1.0 if done else 0.0

        self.ptr = (self.ptr + 1) % self.capacity
        self.size = min(self.size + 1, self.capacity)

    def sample(self, batch_size: int):
        idx = np.random.randint(0, self.size, size=batch_size)
        return (
            self.states[idx],
            self.actions[idx],
            self.rewards[idx],
            self.next_states[idx],
            self.dones[idx],
        )


# -----------------------------
# 3) DQN Network (from scratch)
# -----------------------------
class DQN(nn.Module):
    def __init__(self, state_dim: int, n_actions: int, hidden: int = 128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(state_dim, hidden),
            nn.ReLU(),
            nn.Linear(hidden, hidden),
            nn.ReLU(),
            nn.Linear(hidden, n_actions),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)


# -----------------------------
# 4) Epsilon schedule (linear by steps)
# -----------------------------
class LinearEpsilon:
    def __init__(self, eps_start=1.0, eps_end=0.05, decay_steps=20000):
        self.eps_start = float(eps_start)
        self.eps_end = float(eps_end)
        self.decay_steps = int(decay_steps)

    def value(self, step: int) -> float:
        t = min(max(step, 0), self.decay_steps)
        frac = t / self.decay_steps if self.decay_steps > 0 else 1.0
        return self.eps_start + frac * (self.eps_end - self.eps_start)


# -----------------------------
# 5) Train DQN
# -----------------------------
ARROWS = {0: "↑", 1: "→", 2: "↓", 3: "←"}

@torch.no_grad()
def select_action(q_net: DQN, state: np.ndarray, eps: float, n_actions: int, device: str) -> int:
    if random.random() < eps:
        return random.randrange(n_actions)
    s = torch.from_numpy(state).unsqueeze(0).to(device)  # [1, state_dim]
    q = q_net(s).squeeze(0)  # [n_actions]
    # random tie-break:
    max_q = torch.max(q).item()
    best = (q == max_q).nonzero(as_tuple=False).view(-1).cpu().numpy().tolist()
    return int(random.choice(best))

def train_dqn(
    env: GridWorld,
    total_steps: int = 60000,
    buffer_capacity: int = 50000,
    batch_size: int = 128,
    gamma: float = 0.99,
    lr: float = 1e-3,
    start_learning: int = 1000,
    target_update_every: int = 1000,  # hard update
    eps_start: float = 1.0,
    eps_end: float = 0.05,
    eps_decay_steps: int = 30000,
    grad_clip: float = 10.0,
    seed: int = 0,
):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    device = "cuda" if torch.cuda.is_available() else "cpu"
    state_dim = 2
    n_actions = env.n_actions

    q_net = DQN(state_dim, n_actions).to(device)
    target_net = DQN(state_dim, n_actions).to(device)
    target_net.load_state_dict(q_net.state_dict())
    target_net.eval()

    optimizer = optim.Adam(q_net.parameters(), lr=lr)
    loss_fn = nn.SmoothL1Loss()  # Huber loss
    buffer = ReplayBuffer(buffer_capacity, state_dim)
    eps_sched = LinearEpsilon(eps_start, eps_end, eps_decay_steps)

    s = env.reset()
    ep_return = 0.0
    ep_len = 0
    episode = 0

    for step in range(1, total_steps + 1):
        eps = eps_sched.value(step)
        a = select_action(q_net, s, eps, n_actions, device)
        s2, r, done, _ = env.step(a)

        buffer.push(s, a, r, s2, done)
        ep_return += r
        ep_len += 1
        s = s2

        # Learn
        if buffer.size >= start_learning and buffer.size >= batch_size:
            states, actions, rewards, next_states, dones = buffer.sample(batch_size)

            states_t = torch.from_numpy(states).to(device)          # [B, 2]
            actions_t = torch.from_numpy(actions).to(device)        # [B]
            rewards_t = torch.from_numpy(rewards).to(device)        # [B]
            next_states_t = torch.from_numpy(next_states).to(device)# [B, 2]
            dones_t = torch.from_numpy(dones).to(device)            # [B]

            # Q(s,a)
            q_values = q_net(states_t)                              # [B, A]
            q_sa = q_values.gather(1, actions_t.unsqueeze(1)).squeeze(1)  # [B]

            # target = r + gamma*(1-done)*max_a' Q_target(s', a')
            with torch.no_grad():
                q_next = target_net(next_states_t)                  # [B, A]
                max_q_next = q_next.max(dim=1).values               # [B]
                target = rewards_t + gamma * (1.0 - dones_t) * max_q_next

            loss = loss_fn(q_sa, target)

            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            if grad_clip is not None:
                nn.utils.clip_grad_norm_(q_net.parameters(), grad_clip)
            optimizer.step()

        # Target network hard update
        if step % target_update_every == 0:
            target_net.load_state_dict(q_net.state_dict())

        # Episode end
        if done:
            episode += 1
            if episode % 50 == 0:
                print(
                    f"ep={episode:4d} step={step:6d} "
                    f"return={ep_return:7.3f} len={ep_len:3d} eps={eps:5.3f} "
                    f"device={device}"
                )
            s = env.reset()
            ep_return = 0.0
            ep_len = 0

    return q_net


# -----------------------------
# 6) Visualize learned greedy policy
# -----------------------------
@torch.no_grad()
def greedy_action(q_net: DQN, state: np.ndarray, device: str) -> int:
    s = torch.from_numpy(state).unsqueeze(0).to(device)
    q = q_net(s).squeeze(0)
    return int(torch.argmax(q).item())

def print_policy(env: GridWorld, q_net: DQN):
    device = next(q_net.parameters()).device.type
    for r in range(env.rows):
        row = []
        for c in range(env.cols):
            pos = (r, c)
            if pos == env.start:
                row.append("S")
            elif pos == env.goal:
                row.append("G")
            elif pos == env.pit:
                row.append("P")
            else:
                # temporarily set env.pos to get obs at (r,c)
                old = env.pos
                env.pos = pos
                obs = env._obs()
                env.pos = old
                a = greedy_action(q_net, obs, device)
                row.append(ARROWS[a])
        print(" ".join(row))

def run_greedy_episode(env: GridWorld, q_net: DQN, max_steps: int = 200):
    device = next(q_net.parameters()).device.type
    s = env.reset()
    total = 0.0
    path = [env.pos]
    for _ in range(max_steps):
        a = greedy_action(q_net, s, device)
        s, r, done, _ = env.step(a)
        total += r
        path.append(env.pos)
        if done:
            break
    print("Path:", path)
    print("Return:", total)

In [2]:
# -----------------------------
# 7) Main
# -----------------------------
if __name__ == "__main__":
    env = GridWorld(rows=5, cols=5, start=(0, 0), goal=(4, 4), pit=(3, 3), step_cost=-0.01, max_steps=200)

    q_net = train_dqn(
        env,
        total_steps=60000,
        buffer_capacity=50000,
        batch_size=128,
        gamma=0.99,
        lr=1e-3,
        start_learning=1000,
        target_update_every=1000,
        eps_start=1.0,
        eps_end=0.05,
        eps_decay_steps=30000,
        grad_clip=10.0,
        seed=0,
    )

    print("\nLearned greedy policy (S=start, G=goal, P=pit):")
    print_policy(env, q_net)

    print("\nGreedy rollout:")
    run_greedy_episode(env, q_net)


ep=  50 step=  2809 return=  0.610 len= 40 eps=0.911 device=cuda
ep= 100 step=  5113 return= -1.120 len= 13 eps=0.838 device=cuda
ep= 150 step=  6698 return= -1.120 len= 13 eps=0.788 device=cuda
ep= 200 step=  7845 return= -1.230 len= 24 eps=0.752 device=cuda
ep= 250 step=  8942 return= -1.140 len= 15 eps=0.717 device=cuda
ep= 300 step= 10246 return=  0.920 len=  9 eps=0.676 device=cuda
ep= 350 step= 11242 return=  0.820 len= 19 eps=0.644 device=cuda
ep= 400 step= 12299 return=  0.760 len= 25 eps=0.611 device=cuda
ep= 450 step= 13245 return= -1.160 len= 17 eps=0.581 device=cuda
ep= 500 step= 14126 return=  0.770 len= 24 eps=0.553 device=cuda
ep= 550 step= 15160 return=  0.870 len= 14 eps=0.520 device=cuda
ep= 600 step= 16113 return=  0.900 len= 11 eps=0.490 device=cuda
ep= 650 step= 16895 return=  0.720 len= 29 eps=0.465 device=cuda
ep= 700 step= 17660 return=  0.740 len= 27 eps=0.441 device=cuda
ep= 750 step= 18421 return=  0.810 len= 20 eps=0.417 device=cuda
ep= 800 step= 19247 retur